[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.n_head = num_heads
        self.d_k = int(d_model/num_heads)
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
      b, s_q, d = x_q.shape
      _, s_kv, _ = x_kv.shape
      q = self.wq(x_q).reshape(b, s_q, self.n_head, self.d_k).transpose(1,2)
      k = self.wk(x_kv).reshape(b, s_kv, self.n_head, self.d_k).transpose(1,2)
      v = self.wv(x_kv).reshape(b, s_kv, self.n_head, self.d_k).transpose(1,2)

      score = q @ k.transpose(-2, -1)
      score = score/torch.sqrt(torch.tensor(self.d_k))
      attn_weight = torch.nn.functional.softmax(score, dim=-1)

      output = (attn_weight @ v).transpose(1,2).contiguous().reshape(b, s_q, self.d_model)
      print(output.shape)

      output = self.wo(output)

      return output



In [12]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

torch.Size([2, 6, 64])
Output: torch.Size([2, 6, 64])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
torch.Size([2, 6, 64])
  ✅ [1/4] Output shape (1.7ms)
torch.Size([1, 3, 32])
  ✅ [2/4] Q and KV different lengths (0.7ms)
torch.Size([1, 4, 32])
torch.Size([1, 4, 32])
  ✅ [3/4] No causal mask — all KV affects all Q (40.1ms)
torch.Size([1, 4, 32])
  ✅ [4/4] Gradient flow (34.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (77.0ms total)
  Progress saved. Run status() to see your dashboard.

